In [ ]:
from typing import Annotated, Sequence
from typing import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph import MessagesState,START,END,StateGraph
from langgraph.checkpoint.memory import InMemorySaver

from part1.learn7 import checkpointer


class Main_State(TypedDict):
    messages: Annotated[Sequence[AnyMessage],add_messages]
    query: str
    plan: str
    code: str
    review: str
    is_pass: bool
    retry_count: int

def Supervisor(state: Main_State):
    query = state["query"]
    retry_count = state["retry_count"]
    is_pass = state.get("is_pass",False)

    if is_pass or retry_count >= 3:
        return {"next": "end"}
    elif not state.get("plan"):
        return {"next": "plan"}
    elif not state.get("code"):
        return {"next": "code"}
    else:
        return {"next": "review"}


def Plan(state: Main_State):
    return{}


def Code(state: Main_State):
    return{}

def Review(state: Main_State):
    return{}




graph = StateGraph(Main_State)
graph.add_node("supervisor", Supervisor)
graph.add_node("plan", Plan)
graph.add_node("code", Code)
graph.add_node("review", Review)

graph.add_edge(START, "supervisor")
graph.add_edge("plan", "supervisor")
graph.add_edge("code", "supervisor")
graph.add_edge("review", "supervisor")

graph.add_conditional_edges(
    source="supervisor",
    lambda s: s ["next"],
    {
        "end": END,
        "plan": "plan",
        "code": "code",
        "review": "review"
    }
)
checkpointer = InMemorySaver()
graph.compile(checkpointer=checkpointer)

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "1"}}

    response =  graph.invoke( )
